# ADMM-JOR Algorithm for Traffic Assignment Problem
## Implementing the Parallel Computing Framework from Liu et al. (2024)

This notebook implements the **ADMM-JOR (Alternating Direction Method of Multipliers with Jacobi Over-Relaxation)** algorithm for solving the Deterministic User Equilibrium Traffic Assignment Problem (DUE-TAP), as described in the paper:

> "A novel parallel computing framework for traffic assignment problem: Integrating alternating direction method of multipliers with Jacobi over relaxation method" (Liu et al., 2024)

### Key Features:
- **Link-block decomposition** using edge coloring principle for parallelization
- **JOR iteration enhancement** incorporating historical information for faster convergence
- **Adaptive relaxation factor** selection based on objective function value
- **Convergence monitoring** using relative gap metric

### Algorithm Flow:
1. Load network topology and OD demand matrix
2. Partition links into independent blocks (non-adjacent edges)
3. For each iteration:
   - Update link flows for each block in parallel (ADMM primal step)
   - Update dual variables (Lagrange multipliers)
   - Apply JOR relaxation with historical information
   - Adaptively adjust relaxation factor based on objective function
4. Check convergence using relative gap metric

Let's implement this step by step.

In [ ]:
# ============================================================================
# 1. IMPORT REQUIRED LIBRARIES
# ============================================================================

import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from collections import defaultdict
from typing import Dict, Tuple, List
import time
from tqdm import tqdm

# Configure matplotlib for better visualizations
plt.style.use('seaborn-v0_8-darkgrid')
np.random.seed(42)

print("✓ Libraries imported successfully")
print(f"  - NumPy version: {np.__version__}")
print(f"  - NetworkX version: {nx.__version__}")

## 2. Load Network and OD Data

First, we implement functions to load network topology and OD demand matrices following the structure in `utils.py`.

In [ ]:
def load_network(filepath: str) -> nx.DiGraph:
    """Load network topology from file.
    
    File format:
        # Comments starting with #
        origin_node tail_node capacity free_flow_time
    """
    graph = nx.DiGraph()
    
    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            
            parts = line.split()
            if len(parts) < 4:
                continue
            
            u = int(parts[0])
            v = int(parts[1])
            capacity = float(parts[2])
            free_flow_time = float(parts[3])
            
            graph.add_edge(u, v,
                          capacity=capacity,
                          tempo_fluxo_livre=free_flow_time,
                          fluxo=0.0,
                          custo=free_flow_time,
                          fluxos_por_origem=defaultdict(float))
    
    return graph


def load_od_matrix(filepath: str) -> Dict[Tuple[int, int], float]:
    """Load OD demand matrix from file.
    
    File format:
        # Comments starting with #
        origin destination demand
    """
    viagens = {}
    
    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            
            parts = line.split()
            if len(parts) < 3:
                continue
            
            origin = int(parts[0])
            destination = int(parts[1])
            demand = float(parts[2])
            
            if demand > 0:
                viagens[(origin, destination)] = demand
    
    return viagens


# Example: Create synthetic test network (Sioux Falls-like)
def create_synthetic_network(num_nodes: int = 10) -> Tuple[nx.DiGraph, Dict]:
    """Create a synthetic test network for demonstration."""
    np.random.seed(42)
    graph = nx.DiGraph()
    
    # Add nodes
    for i in range(num_nodes):
        graph.add_node(i)
    
    # Add edges with random properties
    for i in range(num_nodes - 1):
        for j in range(i + 1, min(i + 4, num_nodes)):
            capacity = np.random.uniform(1000, 3000)
            free_flow_time = np.random.uniform(5, 30)
            graph.add_edge(i, j, 
                          capacity=capacity,
                          tempo_fluxo_livre=free_flow_time,
                          fluxo=0.0,
                          custo=free_flow_time,
                          fluxos_por_origem=defaultdict(float))
    
    # Create OD pairs (subset of origins to destinations)
    viagens = {}
    origins = list(range(num_nodes // 2))
    destinations = list(range(num_nodes // 2, num_nodes))
    
    for o in origins:
        for d in destinations:
            if o != d:
                viagens[(o, d)] = np.random.uniform(50, 500)
    
    return graph, viagens


# Create and display test network
print("\n[1] Creating synthetic test network...")
graph_test, viagens_test = create_synthetic_network(num_nodes=10)

print(f"✓ Network created:")
print(f"  - Nodes: {graph_test.number_of_nodes()}")
print(f"  - Links: {graph_test.number_of_edges()}")
print(f"  - OD pairs: {len(viagens_test)}")
print(f"  - Total demand: {sum(viagens_test.values()):.2f} vehicles")

## 3. BPR Link Cost Function

The Bureau of Public Roads (BPR) function models link travel time based on flow:

$$t_a(v_a) = t_0 \left(1 + \alpha \left(\frac{v_a}{Q_a}\right)^\beta\right)$$

where:
- $t_a(v_a)$: travel time on link $a$ with flow $v_a$
- $t_0$: free-flow time
- $Q_a$: link capacity
- $\alpha = 0.15$, $\beta = 4$ (standard parameters)

In [ ]:
# BPR function parameters
BPR_ALPHA = 0.15
BPR_BETA = 4.0


def bpr_cost(free_flow_time: float, flow: float, capacity: float) -> float:
    """Calculate link cost using BPR function."""
    if capacity <= 0:
        return free_flow_time
    ratio = flow / capacity
    return free_flow_time * (1.0 + BPR_ALPHA * (ratio ** BPR_BETA))


def bpr_derivative(free_flow_time: float, flow: float, capacity: float) -> float:
    """Calculate derivative of BPR cost with respect to flow."""
    if capacity <= 0:
        return 0.0
    ratio = flow / capacity
    return free_flow_time * BPR_ALPHA * BPR_BETA * (ratio ** (BPR_BETA - 1)) / capacity


def bpr_integral(free_flow_time: float, flow: float, capacity: float) -> float:
    """Calculate integral of BPR cost function (for objective)."""
    if capacity <= 0:
        return free_flow_time * flow
    ratio = flow / capacity
    return free_flow_time * flow + \
           (BPR_ALPHA * free_flow_time / (BPR_BETA + 1)) * \
           (flow ** (BPR_BETA + 1)) / (capacity ** BPR_BETA)


# Test BPR function
print("\n[2] Testing BPR Cost Function:")
print("─" * 70)
print(f"{'Flow':<15} {'Capacity':<15} {'Free-Flow T':<15} {'Travel Time':<15}")
print("─" * 70)

free_flow = 10
capacity = 2000

for flow in [0, 500, 1000, 1500, 2000, 2500]:
    cost = bpr_cost(free_flow, flow, capacity)
    print(f"{flow:<15.1f} {capacity:<15.1f} {free_flow:<15.1f} {cost:<15.4f}")

print("─" * 70)

## 4. Link Coloring for Block Decomposition

The ADMM method requires partitioning links into independent blocks. We use an edge coloring algorithm to ensure that links within the same block are non-adjacent (don't share vertices).

In [ ]:
def color_edges_greedy(graph: nx.DiGraph) -> Dict[Tuple[int, int], int]:
    """
    Greedy edge coloring to partition links into independent blocks.
    
    Links with the same color don't share vertices (are non-adjacent).
    """
    edges_list = list(graph.edges())
    edge_colors = {}
    node_occupied_colors = defaultdict(set)
    
    for edge in edges_list:
        u, v = edge
        used_colors = node_occupied_colors[u] | node_occupied_colors[v]
        
        # Find smallest color not used by adjacent nodes
        color = 0
        while color in used_colors:
            color += 1
        
        edge_colors[edge] = color
        node_occupied_colors[u].add(color)
        node_occupied_colors[v].add(color)
    
    return edge_colors


def partition_edges_into_blocks(graph: nx.DiGraph) -> Tuple[Dict[int, List], int]:
    """
    Partition edges into independent blocks using edge coloring.
    
    Returns:
        (blocks, num_blocks) where blocks[p] = list of edges in block p
    """
    edge_colors = color_edges_greedy(graph)
    blocks = defaultdict(list)
    
    for edge, color in edge_colors.items():
        blocks[color].append(edge)
    
    num_blocks = max(blocks.keys()) + 1 if blocks else 1
    return dict(blocks), num_blocks


# Apply to test network
print("\n[3] Link Coloring and Block Decomposition:")
print("─" * 70)

blocks_test, num_blocks_test = partition_edges_into_blocks(graph_test)

print(f"Number of blocks: {num_blocks_test}")
print(f"Block sizes: {[len(blocks_test[p]) for p in range(num_blocks_test)]}")

# Visualize block distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Block sizes
block_ids = list(range(num_blocks_test))
block_sizes = [len(blocks_test[p]) for p in block_ids]

ax1.bar(block_ids, block_sizes, color='steelblue', edgecolor='black')
ax1.set_xlabel('Block ID', fontsize=12)
ax1.set_ylabel('Number of Links', fontsize=12)
ax1.set_title('Link Distribution Across Blocks', fontsize=13, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)

# Plot 2: Cumulative blocks
ax2.plot(block_ids, np.cumsum(block_sizes), 'o-', linewidth=2, markersize=8, color='darkblue')
ax2.fill_between(block_ids, 0, np.cumsum(block_sizes), alpha=0.3)
ax2.set_xlabel('Block ID', fontsize=12)
ax2.set_ylabel('Cumulative Links', fontsize=12)
ax2.set_title('Cumulative Link Count', fontsize=13, fontweight='bold')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("─" * 70)

## 5. Network State and Flow Conservation

We maintain the network state including origin-based link flows, dual variables, and link costs.

In [ ]:
def compute_flow_conservation_residual(
    flows: Dict[Tuple[Tuple[int, int], int], float],
    origin: int,
    node: int,
    graph: nx.DiGraph,
    viagens: Dict[Tuple[int, int], float]
) -> float:
    """
    Compute flow conservation constraint residual at a node for an origin.
    
    H_n^o = sum(inflow) - sum(outflow) - g_n^o
    """
    # Incoming flows
    incoming = sum(
        flows.get(((u, node), origin), 0.0)
        for u in graph.predecessors(node)
    )
    
    # Outgoing flows
    outgoing = sum(
        flows.get(((node, v), origin), 0.0)
        for v in graph.successors(node)
    )
    
    # Demand (positive at origin, negative at destination, 0 elsewhere)
    if node == origin:
        demand = sum(viagens.get((origin, d), 0.0) for d in range(graph.number_of_nodes()))
    else:
        demand = -viagens.get((origin, node), 0.0)
    
    residual = incoming - outgoing - demand
    return residual


def update_link_costs(
    graph: nx.DiGraph,
    flows: Dict[Tuple[Tuple[int, int], int], float],
    link_costs: Dict[Tuple[int, int], float],
    origins: List[int]
) -> None:
    """Update all link costs based on current flows using BPR function."""
    for arc in graph.edges():
        # Aggregate flow from all origins
        total_flow = sum(
            flows.get((arc, origin), 0.0)
            for origin in origins
        )
        
        capacity = graph.edges[arc]['capacidade']
        free_flow_time = graph.edges[arc]['tempo_fluxo_livre']
        
        link_costs[arc] = bpr_cost(free_flow_time, total_flow, capacity)


def compute_relative_gap(
    graph: nx.DiGraph,
    flows: Dict[Tuple[Tuple[int, int], int], float],
    link_costs: Dict[Tuple[int, int], float],
    viagens: Dict[Tuple[int, int], float],
    origins: List[int]
) -> float:
    """
    Compute Relative Gap - standard convergence metric for DUE.
    
    Gap = (Z_UE - Z_LB) / Z_UE
    """
    # Current system travel time (Z_UE)
    z_ue = 0.0
    for arc in graph.edges():
        flow = sum(flows.get((arc, origin), 0.0) for origin in origins)
        cost = link_costs.get(arc, 0.0)
        z_ue += flow * cost
    
    # Lower bound using shortest paths (Z_LB)
    z_lb = 0.0
    for origin in origins:
        try:
            _, distances = nx.dijkstra_predecessor_and_distance(
                graph, source=origin, weight='custo'
            )
            for (o, d), demand in viagens.items():
                if o == origin and d in distances:
                    z_lb += demand * distances[d]
        except:
            pass
    
    if z_ue <= 0:
        return 0.0
    
    gap = (z_ue - z_lb) / z_ue
    return max(0.0, gap)


print("✓ Network state and flow conservation functions defined")

## 6. Link-based Subproblem Solver (Gradient Projection)

The augmented Lagrangian for each link-origin pair is optimized using gradient projection.

In [ ]:
def optimize_link_flow(
    arc: Tuple[int, int],
    origin: int,
    flows: Dict[Tuple[Tuple[int, int], int], float],
    lambda_dual: Dict[Tuple[int, int], float],
    graph: nx.DiGraph,
    viagens: Dict[Tuple[int, int], float],
    rho: float,
    step_size: float = 0.01,
    max_inner_iter: int = 20
) -> float:
    """
    Optimize flow on a single link-origin pair using gradient projection.
    
    Minimizes:
        L_rho(v_a^o) = integral_cost + lambda_term + penalty_term
    
    subject to: v_a^o >= 0
    """
    v_ao = flows.get((arc, origin), 0.0)
    u, v = arc
    
    capacity = graph.edges[arc]['capacidade']
    free_flow_time = graph.edges[arc]['tempo_fluxo_livre']
    
    for _ in range(max_inner_iter):
        # Compute residuals at endpoints
        residual_u = compute_flow_conservation_residual(flows, origin, u, graph, viagens)
        residual_v = compute_flow_conservation_residual(flows, origin, v, graph, viagens)
        
        # Gradient calculation
        grad_obj = bpr_derivative(free_flow_time, v_ao, capacity)
        grad_dual = -lambda_dual.get((u, origin), 0.0) - lambda_dual.get((v, origin), 0.0)
        grad_penalty = rho * (residual_u + residual_v)
        
        grad_total = grad_obj + grad_dual + grad_penalty
        
        # Projected gradient step
        v_ao_new = max(0.0, v_ao - step_size * grad_total)
        
        if abs(v_ao_new - v_ao) < 1e-10:
            break
        
        v_ao = v_ao_new
    
    return v_ao


def optimize_block_flows(
    block_edges: List[Tuple[int, int]],
    origin: int,
    flows: Dict[Tuple[Tuple[int, int], int], float],
    flows_iter: Dict[Tuple[Tuple[int, int], int], float],
    lambda_dual: Dict[Tuple[int, int], float],
    graph: nx.DiGraph,
    viagens: Dict[Tuple[int, int], float],
    rho: float
) -> Dict[Tuple[Tuple[int, int], int], float]:
    """
    Optimize flows for all links in a block for a given origin.
    Can be parallelized since links in a block are independent.
    """
    updated_flows = {}
    
    for arc in block_edges:
        v_new = optimize_link_flow(
            arc, origin, flows, lambda_dual, graph, viagens, rho
        )
        updated_flows[(arc, origin)] = v_new
    
    return updated_flows


print("✓ Link optimization functions defined")

## 7. ADMM-JOR Main Algorithm

Now we implement the complete ADMM-JOR algorithm combining all components.

In [ ]:
def compute_objective_function(
    graph: nx.DiGraph,
    flows: Dict[Tuple[Tuple[int, int], int], float],
    viagens: Dict[Tuple[int, int], float],
    origins: List[int]
) -> float:
    """
    Compute objective function value (total system travel time).
    
    f(v) = sum_a integral_0^{v_a} t_a(w) dw
    """
    total_cost = 0.0
    
    for arc in graph.edges():
        capacity = graph.edges[arc]['capacidade']
        free_flow_time = graph.edges[arc]['tempo_fluxo_livre']
        
        for origin in origins:
            v_ao = flows.get((arc, origin), 0.0)
            if v_ao > 0:
                total_cost += bpr_integral(free_flow_time, v_ao, capacity)
    
    return total_cost


def admm_jor_solver(
    graph: nx.DiGraph,
    viagens: Dict[Tuple[int, int], float],
    rho: float = 1.0,
    omega: float = 1.2,
    max_iterations: int = 200,
    tolerance: float = 1e-6,
    adaptive_omega: bool = True,
    verbose: bool = True
) -> Tuple[Dict, List[float], int]:
    """
    Solve DUE-TAP using ADMM-JOR algorithm.
    
    Parameters:
        graph: Network topology
        viagens: OD demand matrix
        rho: Penalty parameter
        omega: Initial relaxation factor (0, 2)
        max_iterations: Maximum iterations
        tolerance: Convergence tolerance (gap)
        adaptive_omega: Whether to adaptively adjust omega
        verbose: Print progress
    
    Returns:
        (flows, gaps, iterations)
    """
    start_time = time.time()
    
    # Initialize
    origins = sorted(set(o for o, d in viagens.keys()))
    destinations = sorted(set(d for o, d in viagens.keys()))
    
    # Create blocks
    blocks, num_blocks = partition_edges_into_blocks(graph)
    
    # Initialize flows and dual variables
    flows = {}  # (arc, origin) -> flow
    flows_iter = {}  # intermediate ADMM flows
    lambda_dual = {}  # (node, origin) -> dual variable
    link_costs = {}
    
    for arc in graph.edges():
        for origin in origins:
            flows[(arc, origin)] = 0.0
            flows_iter[(arc, origin)] = 0.0
        link_costs[arc] = graph.edges[arc]['tempo_fluxo_livre']
    
    for node in graph.nodes():
        for origin in origins:
            lambda_dual[(node, origin)] = 0.0
    
    if verbose:
        print("\n" + "=" * 80)
        print("ADMM-JOR ALGORITHM FOR DUE-TAP")
        print("=" * 80)
        print(f"Network: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} links")
        print(f"Blocks: {num_blocks} (sizes: {[len(blocks[p]) for p in range(num_blocks)]})")
        print(f"Origins: {len(origins)}, OD pairs: {len(viagens)}")
        print(f"Parameters: rho={rho}, omega={omega}, adaptive={adaptive_omega}")
        print("=" * 80)
    
    gaps = []
    omega_history = [omega]
    
    for iteration in range(max_iterations):
        # Update link costs
        update_link_costs(graph, flows, link_costs, origins)
        
        # ADMM Step 1: Update flows for each block (parallelizable)
        for p in range(num_blocks):
            block_edges = blocks[p]
            
            for origin in origins:
                updated = optimize_block_flows(
                    block_edges, origin, flows, flows_iter, 
                    lambda_dual, graph, viagens, rho
                )
                
                # Store ADMM updates
                for (arc, _), v_new in updated.items():
                    flows_iter[(arc, origin)] = v_new
        
        # ADMM Step 2: Apply JOR relaxation globally
        for (arc, origin) in flows_iter:
            v_new_admm = flows_iter[(arc, origin)]
            v_old = flows.get((arc, origin), 0.0)
            
            # JOR update: v^{k+1} = omega * v_new_admm + (1 - omega) * v_old
            flows[(arc, origin)] = omega * v_new_admm + (1.0 - omega) * v_old
        
        # ADMM Step 3: Update dual variables
        for node in graph.nodes():
            for origin in origins:
                residual = compute_flow_conservation_residual(
                    flows, origin, node, graph, viagens
                )
                lambda_dual[(node, origin)] += rho * residual
        
        # Compute convergence metric
        update_link_costs(graph, flows, link_costs, origins)
        gap = compute_relative_gap(graph, flows, link_costs, viagens, origins)
        gaps.append(gap)
        
        # Adaptive omega adjustment
        if adaptive_omega and iteration > 0:
            if gaps[-1] < gaps[-2]:
                omega = min(omega * 1.02, 2.0)
            else:
                omega = max(omega * 0.98, 1.0)
            omega_history.append(omega)
        
        if verbose and (iteration % 20 == 0 or iteration < 5 or iteration == max_iterations - 1):
            elapsed = time.time() - start_time
            print(f"Iter {iteration:4d} | Gap: {gap:.6e} | Omega: {omega:.4f} | Time: {elapsed:.2f}s")
        
        if gap < tolerance:
            if verbose:
                print(f"\n✓ Converged at iteration {iteration}")
            break
    
    elapsed = time.time() - start_time
    
    if verbose:
        print("=" * 80)
        print(f"Iterations: {iteration + 1} / {max_iterations}")
        print(f"Final gap: {gaps[-1]:.6e}")
        print(f"Computation time: {elapsed:.2f}s")
        print("=" * 80 + "\n")
    
    return flows, gaps, iteration + 1


print("✓ ADMM-JOR solver implemented")

## 8. Run ADMM-JOR on Test Network

Now let's solve the traffic assignment problem using both standard ADMM and ADMM-JOR.

In [ ]:
print("\n[4] Solving Test Network with ADMM-JOR\n")

# Solve with standard ADMM (omega = 1.0)
print("Solution 1: Standard ADMM (omega = 1.0, no JOR relaxation)")
print("─" * 80)
flows_admm, gaps_admm, iters_admm = admm_jor_solver(
    graph_test, viagens_test,
    rho=0.5,
    omega=1.0,  # No JOR relaxation
    max_iterations=200,
    tolerance=1e-6,
    adaptive_omega=False,
    verbose=True
)

# Solve with ADMM-JOR (omega > 1.0 with fixed relaxation)
print("\nSolution 2: ADMM-JOR with Fixed Relaxation (omega = 1.3)")
print("─" * 80)
flows_jor_fixed, gaps_jor_fixed, iters_jor_fixed = admm_jor_solver(
    graph_test, viagens_test,
    rho=0.5,
    omega=1.3,  # JOR relaxation factor
    max_iterations=200,
    tolerance=1e-6,
    adaptive_omega=False,
    verbose=True
)

# Solve with ADMM-JOR adaptive (omega adaptively adjusted)
print("\nSolution 3: ADMM-JOR with Adaptive Relaxation")
print("─" * 80)
flows_jor_adaptive, gaps_jor_adaptive, iters_jor_adaptive = admm_jor_solver(
    graph_test, viagens_test,
    rho=0.5,
    omega=1.2,
    max_iterations=200,
    tolerance=1e-6,
    adaptive_omega=True,
    verbose=True
)

## 9. Performance Comparison and Visualization

Compare convergence behavior across the three algorithms.

In [ ]:
print("\n[5] Performance Comparison\n")
print("=" * 80)
print(f"{'Algorithm':<35} {'Iterations':<15} {'Final Gap':<20}")
print("=" * 80)
print(f"{'Standard ADMM (ω=1.0)':<35} {iters_admm:<15} {gaps_admm[-1]:.6e}")
print(f"{'ADMM-JOR Fixed (ω=1.3)':<35} {iters_jor_fixed:<15} {gaps_jor_fixed[-1]:.6e}")
print(f"{'ADMM-JOR Adaptive':<35} {iters_jor_adaptive:<15} {gaps_jor_adaptive[-1]:.6e}")
print("=" * 80)

# Calculate improvements
improvement_fixed = (iters_admm - iters_jor_fixed) / iters_admm * 100
improvement_adaptive = (iters_admm - iters_jor_adaptive) / iters_admm * 100

print(f"\nIteration reduction:")
print(f"  Fixed relaxation:    {improvement_fixed:+.2f}%")
print(f"  Adaptive relaxation: {improvement_adaptive:+.2f}%")
print("=" * 80 + "\n")

# Convergence plot
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Plot 1: Convergence curves
ax = axes[0]
ax.semilogy(range(len(gaps_admm)), gaps_admm, 'o-', linewidth=2.5, 
            markersize=5, label='ADMM (ω=1.0)', color='#1f77b4')
ax.semilogy(range(len(gaps_jor_fixed)), gaps_jor_fixed, 's-', linewidth=2.5,
            markersize=5, label='ADMM-JOR Fixed (ω=1.3)', color='#ff7f0e')
ax.semilogy(range(len(gaps_jor_adaptive)), gaps_jor_adaptive, '^-', linewidth=2.5,
            markersize=5, label='ADMM-JOR Adaptive', color='#2ca02c')

ax.set_xlabel('Iteration', fontsize=12, fontweight='bold')
ax.set_ylabel('Relative Gap', fontsize=12, fontweight='bold')
ax.set_title('Convergence Comparison: Relative Gap', fontsize=13, fontweight='bold')
ax.legend(fontsize=11, loc='upper right')
ax.grid(True, alpha=0.3)

# Plot 2: Iteration performance comparison
ax = axes[1]
algorithms = ['ADMM\n(ω=1.0)', 'ADMM-JOR\nFixed (ω=1.3)', 'ADMM-JOR\nAdaptive']
iterations = [iters_admm, iters_jor_fixed, iters_jor_adaptive]
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

bars = ax.bar(algorithms, iterations, color=colors, edgecolor='black', linewidth=1.5, alpha=0.8)

# Add value labels on bars
for bar, val in zip(bars, iterations):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(val)}',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_ylabel('Number of Iterations', fontsize=12, fontweight='bold')
ax.set_title('Iteration Comparison', fontsize=13, fontweight='bold')
ax.set_ylim(0, max(iterations) * 1.15)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("Plots generated successfully!")

## 10. Link Flow Analysis and Solution Properties

Extract and analyze the final solution.

In [ ]:
print("\n[6] Solution Analysis\n")

# Extract aggregate link flows
def get_aggregate_flows(flows, graph, origins):
    link_flows = {}
    for arc in graph.edges():
        total_flow = sum(flows.get((arc, origin), 0.0) for origin in origins)
        link_flows[arc] = total_flow
    return link_flows

link_flows_jor = get_aggregate_flows(flows_jor_adaptive, graph_test, 
                                      list(set(o for o, _ in viagens_test.keys())))

# Calculate network statistics
origins_test = sorted(set(o for o, d in viagens_test.keys()))
destinations_test = sorted(set(d for o, d in viagens_test.keys()))

print("Network Statistics:")
print("─" * 80)
print(f"Total network flow (all links): {sum(link_flows_jor.values()):.2f} vehicles")
print(f"Total demand: {sum(viagens_test.values()):.2f} vehicles")
print(f"Links with positive flow: {sum(1 for f in link_flows_jor.values() if f > 0.01)}")
print(f"Links with zero flow: {sum(1 for f in link_flows_jor.values() if f <= 0.01)}")

# Find most congested links
print("\n" + "─" * 80)
print("Top 10 Most Congested Links:")
print("─" * 80)
print(f"{'Link':<15} {'Flow':<15} {'Capacity':<15} {'V/C Ratio':<15}")
print("─" * 80)

sorted_links = sorted(link_flows_jor.items(), key=lambda x: x[1], reverse=True)

for (u, v), flow in sorted_links[:10]:
    capacity = graph_test.edges[(u, v)]['capacidade']
    vc_ratio = flow / capacity if capacity > 0 else 0
    print(f"({u:2d}→{v:2d})     {flow:<14.2f} {capacity:<14.2f} {vc_ratio:<14.4f}")

print("─" * 80 + "\n")

# V/C ratio histogram
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: V/C ratio distribution
vc_ratios = [link_flows_jor[arc] / graph_test.edges[arc]['capacidade'] 
             for arc in graph_test.edges()]
vc_ratios = [min(r, 2.0) for r in vc_ratios]  # Cap at 2.0 for visualization

ax = axes[0]
ax.hist(vc_ratios, bins=20, color='steelblue', edgecolor='black', alpha=0.7)
ax.axvline(np.mean(vc_ratios), color='red', linestyle='--', linewidth=2, label=f'Mean: {np.mean(vc_ratios):.3f}')
ax.set_xlabel('Volume/Capacity Ratio', fontsize=12, fontweight='bold')
ax.set_ylabel('Number of Links', fontsize=12, fontweight='bold')
ax.set_title('V/C Ratio Distribution', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)

# Plot 2: Flow distribution
flows_list = list(link_flows_jor.values())
flows_list = [f for f in flows_list if f > 0.1]  # Only positive flows

ax = axes[1]
ax.hist(flows_list, bins=20, color='darkgreen', edgecolor='black', alpha=0.7)
ax.set_xlabel('Link Flow (vehicles)', fontsize=12, fontweight='bold')
ax.set_ylabel('Number of Links', fontsize=12, fontweight='bold')
ax.set_title('Flow Distribution', fontsize=13, fontweight='bold')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("Analysis plots generated!")

## 11. Key Findings and Algorithm Properties

### Algorithm Features Implemented:

1. **Link-Block Decomposition**: Using edge coloring principle to partition links into independent blocks that can be solved in parallel

2. **JOR Relaxation**: The core ADMM-JOR enhancement applies a relaxation factor $\omega$ to incorporate historical information:
   $$v^{k+1} = \omega \tilde{v}^k + (1-\omega) v^k$$
   where $\tilde{v}^k$ is from the ADMM step

3. **Convergence Metric**: Relative gap monitoring
   $$|RG| = \left|1 - \frac{Z_{LB}}{Z_{UE}}\right|$$

4. **Adaptive Relaxation** (optional): Automatically adjusts $\omega$ based on objective function improvement

### Convergence Properties:

In [ ]:
print("\n[7] ALGORITHM SUMMARY\n")

summary_text = """
╔════════════════════════════════════════════════════════════════════════════╗
║                     ADMM-JOR ALGORITHM IMPLEMENTATION                      ║
╠════════════════════════════════════════════════════════════════════════════╣
║                                                                            ║
║  Paper: Liu et al. (2024) - "A novel parallel computing framework for     ║
║         traffic assignment problem: Integrating ADMM with JOR"            ║
║                                                                            ║
║  ▪ Link Blocks: {} blocks (non-adjacent edges via coloring)
║                                                                            ║
║  ▪ Algorithm Steps per Iteration:                                         ║
║    1. Update link flows for each block (parallelizable)                  ║
║    2. Update dual variables (Lagrange multipliers)                        ║
║    3. Apply JOR relaxation with factor ω ∈ (0,2)                         ║
║    4. Optionally adapt ω based on objective function value               ║
║                                                                            ║
║  ▪ JOR Enhancement (core contribution):                                   ║
║    v^(k+1) = ω·ṽ^k + (1-ω)·v^k                                           ║
║                                                                            ║
║  ▪ Performance Benefits:                                                  ║
║    • Faster convergence than standard ADMM                                ║
║    • Maintains excellent parallelization properties                       ║
║    • Requires minimal additional computational overhead                   ║
║                                                                            ║
║  ▪ Convergence Guarantee:                                                 ║
║    Proven under contractive-type methods framework                        ║
║    when ω ∈ (0,2)                                                         ║
║                                                                            ║
╚════════════════════════════════════════════════════════════════════════════╝
""".format(num_blocks_test)

print(summary_text)

print("\nRESULTS OBTAINED:")
print("─" * 80)
print(f"Test Network: {graph_test.number_of_nodes()} nodes, {graph_test.number_of_edges()} links")
print(f"Demand: {len(viagens_test)} OD pairs, {sum(viagens_test.values()):.0f} vehicles")
print()
print("Comparison:")
print(f"  Standard ADMM:           {iters_admm} iterations, gap={gaps_admm[-1]:.2e}")
print(f"  ADMM-JOR (ω=1.3):        {iters_jor_fixed} iterations, gap={gaps_jor_fixed[-1]:.2e}")
print(f"  ADMM-JOR (adaptive):     {iters_jor_adaptive} iterations, gap={gaps_jor_adaptive[-1]:.2e}")
print()
print(f"Improvement from adaptive relaxation: {improvement_adaptive:.1f}% fewer iterations")
print("─" * 80)

## 12. Usage with Real Networks

To use this implementation with real transportation networks (Sioux Falls, Anaheim, Chicago-Sketch), load them as follows: